In [ ]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [ ]:
# Load PROMISE dataset
# Common column names are: 'RequirementText', 'class', or 'label'
promise_df = pd.read_csv('../data/raw/promise_nfr.csv')  # adjust filename

print("Shape:", promise_df.shape)
print("\nColumns:", promise_df.columns.tolist())
print("\nFirst 5 rows:")
promise_df.head()

In [ ]:
print("Label distribution in PROMISE:")
print(promise_df['class'].value_counts())  # adjust column name if different

# Visualize
promise_df['class'].value_counts().plot(kind='bar', figsize=(12,5), title='PROMISE Label Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# PURE dataset — may come as multiple files or one CSV
pure_df = pd.read_csv('../data/raw/pure_dataset.csv')  # adjust filename

print("Shape:", pure_df.shape)
print("\nColumns:", pure_df.columns.tolist())
pure_df.head()

In [ ]:
fr_nfr_df = pd.read_csv('../data/raw/fr_nfr_dataset.csv')  # adjust filename

print("Shape:", fr_nfr_df.shape)
print("\nColumns:", fr_nfr_df.columns.tolist())
fr_nfr_df.head()

In [ ]:
# ---- Standardize PROMISE ----
# Typical PROMISE columns: 'RequirementText', 'class'
promise_clean = pd.DataFrame({
    'text': promise_df['RequirementText'],  # change if your column name differs
    'label': promise_df['class'],           # change if your column name differs
    'source': 'PROMISE'
})

# ---- Standardize PURE ----
# Typical PURE columns may vary — inspect yours and adjust
pure_clean = pd.DataFrame({
    'text': pure_df['Text'],          # adjust
    'label': pure_df['Label'],        # adjust  
    'source': 'PURE'
})

# ---- Standardize FR_NFR ----
fr_nfr_clean = pd.DataFrame({
    'text': fr_nfr_df['RequirementText'],   # adjust
    'label': fr_nfr_df['RequirementType'],  # adjust
    'source': 'FR_NFR'
})

print("PROMISE labels:", promise_clean['label'].unique())
print("PURE labels:", pure_clean['label'].unique())
print("FR_NFR labels:", fr_nfr_clean['label'].unique())

In [ ]:
# Define your master label mapping
# Look at the unique labels printed above and map them here
label_mapping = {
    # PROMISE labels (common ones)
    'F'  : 'FR',
    'FR' : 'FR',
    'A'  : 'NFR_Availability',
    'FT' : 'NFR_FaultTolerance',
    'L'  : 'NFR_Legal',
    'LF' : 'NFR_LookAndFeel',
    'MN' : 'NFR_Maintainability',
    'O'  : 'NFR_Operational',
    'PE' : 'NFR_Performance',
    'PO' : 'NFR_Portability',
    'SC' : 'NFR_Scalability',
    'SE' : 'NFR_Security',
    'US' : 'NFR_Usability',
    
    # PURE / FR_NFR labels (common ones)
    'Functional'    : 'FR',
    'NonFunctional' : 'NFR_Other',
    'NFR'           : 'NFR_Other',
    '0'             : 'FR',
    '1'             : 'NFR_Other',
}

# Apply mapping to each dataset
promise_clean['label'] = promise_clean['label'].map(label_mapping)
pure_clean['label']    = pure_clean['label'].map(label_mapping)
fr_nfr_clean['label'] = fr_nfr_clean['label'].map(label_mapping)

# Drop any rows where label couldn't be mapped (NaN)
promise_clean.dropna(subset=['label'], inplace=True)
pure_clean.dropna(subset=['label'], inplace=True)
fr_nfr_clean.dropna(subset=['label'], inplace=True)

In [ ]:
# Merge all into one master dataframe
master_df = pd.concat([promise_clean, pure_clean, fr_nfr_clean], ignore_index=True)

print("Total records:", len(master_df))
print("\nLabel distribution:")
print(master_df['label'].value_counts())

master_df.sample(10)  # inspect random samples

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)         # remove extra whitespace
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # remove non-ASCII characters
    return text

master_df['text'] = master_df['text'].apply(clean_text)

# Remove empty rows
master_df = master_df[master_df['text'].str.len() > 10]

print("After cleaning:", len(master_df), "records")

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
master_df['label_id'] = le.fit_transform(master_df['label'])

# Save the label mapping so you can decode predictions later
label2id = {label: idx for idx, label in enumerate(le.classes_)}
id2label = {idx: label for label, idx in label2id.items()}

print("Label encoding map:")
for k, v in label2id.items():
    print(f"  {k} → {v}")

In [ ]:
master_df.to_csv('../data/processed/master_dataset.csv', index=False)

# Also save label maps
import json
with open('../data/processed/label2id.json', 'w') as f:
    json.dump(label2id, f, indent=2)
with open('../data/processed/id2label.json', 'w') as f:
    json.dump(id2label, f, indent=2)

print("✅ Saved master_dataset.csv and label maps!")
print(f"Total samples: {len(master_df)}")
print(f"Number of classes: {len(label2id)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Label distribution
master_df['label'].value_counts().plot(kind='bar', ax=axes[0], title='Final Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

# Source distribution
master_df['source'].value_counts().plot(kind='pie', ax=axes[1], title='Data Source Distribution', autopct='%1.1f%%')

plt.tight_layout()
plt.savefig('../data/processed/dataset_overview.png')
plt.show()
print("✅ Preprocessing complete!")